In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

import plotly.io as pio

pio.templates.default = "plotly_white"

In [ ]:
template_df = pd.read_csv("../data/mhcii_tcr_templates.csv", index_col="pdb_id")
pdb_ids = template_df.index.to_list()

In [ ]:
ensembles_dir = "../data/mhcii_tcr_ensembles/"
sampling_method = "ensemble/annealing"
docking_methods = ["haddock3_rigidbody", "haddock3_flexref", "haddock3_rigidbody_less_restrained"]

In [ ]:
df = None

for pdb_id in pdb_ids:
    for docking_method in docking_methods:
        ensemble_dir = os.path.join(pdb_id, sampling_method, docking_method)
        csv_fp = os.path.join(ensembles_dir, ensemble_dir, "stats.csv")
        if not os.path.exists(csv_fp):
            continue
        ensemble_df = pd.read_csv(csv_fp, index_col="model_id")
        ensemble_df.drop("pdb_fp", axis=1, inplace=True)
        ensemble_df["pdb_id"] = pdb_id
        ensemble_df["docking_method"] = docking_method
        if df is None:
            df = ensemble_df
        else:
            df = pd.concat([df, ensemble_df])

In [ ]:
df

In [ ]:
metrics = ["rmsd", "dockq", "dockq_fnat", "binding_core_sasa"]
pdb_ids = template_df.index.to_list()
colors = ["blue", "limegreen", "crimson", "green"]

In [ ]:
for metric in metrics:
    fig = go.Figure()
    for i, pdb_id in enumerate(pdb_ids):
        pdb_df = df[df.pdb_id == pdb_id]
        for docking_method, color in zip(docking_methods, colors):
            ens_df = pdb_df[pdb_df.docking_method == docking_method]
            # print(docking_method, color, ens_df)
            fig.add_trace(
                go.Box(
                    y=ens_df[metric].values,
                    name=pdb_id,
                    offsetgroup=docking_method,
                    legendgroup=docking_method,
                    showlegend=(i == 0),
                    line=dict(color=color),
                    marker=dict(color=color)
                )
            )
    fig.update_xaxes(showticklabels=True)
    fig.update_layout(title=metric, boxmode="group")
    fig.show()

In [ ]:
from scipy.stats import spearmanr

fig = go.Figure()
pdb_df = df[df.pdb_id == "6BGA"]

for docking_method, color in zip(docking_methods, colors):
    method_df = pdb_df[pdb_df.docking_method == docking_method]
    method_df = method_df[method_df.index.str.endswith("1")]
    fig.add_trace(
        go.Scatter(
            x=method_df.dockq_fnat,
            y=method_df.binding_core_sasa,
            mode="markers",
            marker=dict(color=color),
            name=docking_method
        )
    )
    print(spearmanr(method_df.dockq_fnat, method_df.binding_core_sasa))

fig.show()